# TripAdvisor Sentiment Analysis

## 0. Import Necessary Libraries

In [ ]:
import nltk
import nltk.corpus
import pandas as pd
import os
import glob

## 1. Load Data and Transform Data Types

In [ ]:
def read_and_transform_data(file_name):
    '''
    Transform data before EDA
    '''
    # Read data
    data = pd.read_excel(file_name)
    
    # Drop previous index
    data = data.drop(['Unnamed: 0'], axis=1)
    
    # Change string to datetime
    data['write_date'] = pd.to_datetime(data.write_date)
    data['date_dep'] = pd.to_datetime(data.date_dep)
    
    # Filter after 2014 only
    data = data[data['write_date'] >= '2014-1-1']
    
    return data

In [ ]:
path = 'Koh Larn'
files = glob.glob(path + '/*.xlsx')
df = pd.DataFrame()
for file in files:
    df_file = read_and_transform_data(file)
    df = pd.concat([df, df_file])

In [ ]:
# Preview dataset
df.head()

In [ ]:
# Check data type and null values
df.info()

## 2. Exploratory Data Analysis

### 2.1 Number of Reviews by Reviewer

In [ ]:
df_reviewers = pd.DataFrame(df.groupby(['review_name'])['review_name'].count())
df_reviewers = df_reviewers.rename(columns={"review_name": "count"})
df_reviewers = df_reviewers.sort_values(by=['count'], ascending=False)

In [ ]:
df_reviewers.describe()

### 2.2 Reviewer Location Distribution

In [ ]:
df_address = pd.DataFrame(df.groupby(['address'])['address'].count())
df_address = df_address.rename(columns={"address": "count"})
df_address = df_address.sort_values(by=['count'], ascending=False)

In [ ]:
df_address.head(10)

## 3. Sentiment Analysis Using VADER

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
sid_obj = SentimentIntensityAnalyzer()

In [ ]:
compound_score = []
sentiment_list = []

for sentence in df['text_review']:
    
    sentiment_dict = sid_obj.polarity_scores(sentence)
    
    if sentiment_dict['compound'] >= 0.05 :
        result = "Positive"
 
    elif sentiment_dict['compound'] <= - 0.05 :
        result = "Negative"
 
    else :
        result = "Neutral"

    compound_score.append(sentiment_dict['compound'])
    sentiment_list.append(result)

In [ ]:
df['compound_score'] = compound_score
df['sentiment'] = sentiment_list

In [ ]:
df

## 4. Time-Based Features

In [ ]:
df['year'] = df['write_date'].dt.year
df['month_year'] = df['write_date'].dt.to_period('M')
df['month'] = df['write_date'].dt.month

In [ ]:
df

## 5. Export Sentiment Results

In [ ]:
df.to_excel(f'{path}.xlsx')

## 6. Text Preprocessing and Related-Word Exploration

In [ ]:
# importing stopwords from nltk library
from nltk import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

stopwords = set(stopwords.words('english'))
group_of_word = []

for sentence in df_split_sentence['sentence']:
    text = word_tokenize(sentence.lower())
    stopwords_clean = [x for x in text if x not in stopwords]
    related_word = []
    for token in stopwords_clean:
        if nltk.pos_tag([token])[0][1] == "NN":
            related_word.append(nltk.pos_tag([token])[0][0])
    group_of_word.append((related_word))

In [ ]:
df_split_sentence['related_word'] = group_of_word

In [ ]:
df_split_sentence

## 8. Year-Based Review Analysis

In [ ]:
year_list = [i for i in range(2014, 2024)]
year_list

In [ ]:
df_year = df[df['year'] == 2014]

## 9. Word-Frequency Analysis

In [ ]:
# importing stopwords from nltk library
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
import collections
import operator

lemmatizer = WordNetLemmatizer()

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download("punkt")

sentiment = 'Negative'

stopwords = set(stopwords.words('english'))

for year in year_list:

    
    df_year = df[df['year'] == year]
    df_year = df_year[df_year['sentiment'] == sentiment]
    text_clean = []
    
    for review in df_year['text_review']:
        text = word_tokenize(review.lower())
        
        new_words= [word for word in text if word.isalnum()]
        
        stopwords_clean = [x for x in new_words if x not in stopwords]
        
        
        for word in stopwords_clean:
            word_clean = lemmatizer.lemmatize(word)
            text_clean.append(word_clean)
    print(year)
    frequency = dict(collections.Counter(text_clean))
    sorted_dict = dict(sorted(frequency.items(), key=operator.itemgetter(1), reverse=True))
    df_result = pd.DataFrame(list(sorted_dict.items()), columns=['word', 'Freq'])
    df_result.head(10).to_excel(f'freq_{year}_{sentiment}.xlsx')
    print("*********************************************")

In [ ]:
# new_name = file_name.split(".")[0] + "_result." +  file_name.split(".")[1]
new_name = path + '.xlsx'
df_split_sentence.to_excel(new_name)

In [ ]:
df_year = df
df_year = df_year[df_year['sentiment'] == sentiment]
text_clean = []

for review in df_year['text_review']:
    text = word_tokenize(review.lower())

    new_words= [word for word in text if word.isalnum()]

    stopwords_clean = [x for x in new_words if x not in stopwords]


    for word in stopwords_clean:
        word_clean = lemmatizer.lemmatize(word)
        text_clean.append(word_clean)
frequency = dict(collections.Counter(text_clean))
sorted_dict = dict(sorted(frequency.items(), key=operator.itemgetter(1), reverse=True))
df_result = pd.DataFrame(list(sorted_dict.items()), columns=['word', 'Freq'])
df_result.head(10).to_excel(f'freq_all_{sentiment}.xlsx')
print("*********************************************")